# Episode 1 — Environment Setup & First Look at the Data

**Course:** Production RAG — YouTube Series  
**GitHub branch:** `episode/01`

This notebook accompanies Episode 1. It walks through:
1. Verifying your environment is correctly configured
2. Your first OpenAI API call
3. Loading a DHS PDF and inspecting raw text
4. Seeing the hallucination problem that RAG solves

---
> **Before you start:** Copy `.env.example` to `.env` and fill in your `OPENAI_API_KEY`.

## 1. Environment check

In [ ]:
# Verify Python version
import sys
print(f'Python {sys.version}')
assert sys.version_info >= (3, 12), 'Need Python 3.12+'
print('✅ Python version OK')

In [ ]:
# Load environment variables from .env
from dotenv import load_dotenv
import os

loaded = load_dotenv()
print(f'Loaded .env: {loaded}')

api_key = os.getenv('OPENAI_API_KEY', '')
if not api_key or not api_key.startswith('sk-'):
    print('❌ OPENAI_API_KEY not set or invalid. Check your .env file.')
else:
    print(f'✅ OPENAI_API_KEY found: sk-...{api_key[-4:]}')

## 2. First OpenAI call — smoke test

In [ ]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)
response = llm.invoke('Say hello in exactly one sentence.')
print(response.content)
print('✅ OpenAI API working')

## 3. The hallucination problem

In [ ]:
# Ask the LLM about a specific DHS statistic — watch it hallucinate
question = (
    'What is the exact maternal mortality ratio per 100,000 live births '
    'in Nigeria according to the 2021 DHS report? '
    'Give me the precise number from the report.'
)

response = llm.invoke(question)
print('=== LLM Answer (no RAG) ===')
print(response.content)
print()
print('🚨 Note: The model does not know if this is correct.')
print('   The actual figure is 512 per 100,000 (2021 Nigeria DHS).')
print('   RAG grounds this answer in the actual document. That is what we build.')

## 4. Load a PDF (if you have one in data/raw/)

In [ ]:
from pathlib import Path

data_dir = Path('../data/raw')
pdfs = list(data_dir.glob('*.pdf'))

if not pdfs:
    print('No PDFs found in data/raw/')
    print('Follow the instructions in data/README.md to download DHS reports.')
else:
    print(f'Found {len(pdfs)} PDF(s):')
    for pdf in pdfs:
        print(f'  {pdf.name}  ({pdf.stat().st_size / 1024 / 1024:.1f} MB)')

In [ ]:
# Load the first PDF and inspect raw text
# This is the raw output BEFORE cleaning — note the artefacts

import sys
sys.path.insert(0, '../src')

if pdfs:
    from rag.ingestion.loader import load_pdf
    
    pdf_path = pdfs[0]
    pages = load_pdf(pdf_path)
    
    print(f'Loaded {len(pages)} pages from {pdf_path.name}')
    print()
    print('=== First page raw text (first 2000 chars) ===')
    print(pages[0].text[:2000])
    print('...')
    print()
    print('Notice: broken words, footnote numbers, repeated headers.')
    print('Episode 2 builds the cleaner that fixes all of this.')

## What's next

**Episode 2** — Document ingestion and chunking
- We fix all the PDF artefacts you saw above
- Compare three chunking strategies on the same document
- Attach metadata (country, year, report type) to every chunk
- Print summary statistics and see why chunk quality matters

**GitHub branch:** `episode/02`

---
*Questions? Drop them in the YouTube comments or open a GitHub Discussion.*